# 聚合 real-video VAE latent probe shards

该 Notebook 只负责把 `shard_runs` 中的阶段二分片结果聚合为可供阶段三使用的正式结果包。它不执行 VAE runner。

In [ ]:
# 可修改配置区
REPO_URL = "https://github.com/RICHAAARC/TSTW-VW.git"  # Colab 冷启动时使用的仓库 clone URL
REPO_DIR = "/content/TSTW-VW"  # Colab 本地仓库根目录
GIT_REF = "explicit-sync"  # 要拉取的项目分支
DRIVE_MOUNT = "/content/drive"  # Google Drive 挂载点
DRIVE_ROOT = "/content/drive/MyDrive"  # Google Drive MyDrive 根目录
LOCAL_RUNTIME_ROOT = "/content/TSTW_runtime"  # 本次 Colab session 的本地运行根目录
SHARD_ROOTS = []  # 需要聚合的 shard 结果目录或 zip 路径列表, 建议填入 TSTW/results/real_video_vae_latent_probe/shard_runs 下的完整路径
MERGED_RUN_ROOT = f"{LOCAL_RUNTIME_ROOT}/runs/real_video_vae_latent_probe_shard_aggregated"  # 聚合后的本地 run_root
LOCAL_FAMILY_ROOT = f"{LOCAL_RUNTIME_ROOT}/families/real_video_vae_latent_probe/shard_aggregated"  # 聚合包先写入的本地 family 父目录
PROTOCOL_CONFIG = "configs/protocol/real_video_vae_latent_probe.json"  # 阶段二 fixed-FPR 协议配置
MECHANISM_CONFIG = "configs/protocol/stage2_mechanism_gate.json"  # 阶段二 mechanism gate 配置
RUNTIME_PROFILE = "formal"  # 聚合结果登记使用的 runtime profile
EXCLUDE_LARGE_INTERMEDIATE_LATENTS = True  # 聚合包默认瘦身, 不重复打包大型 latent/video 中间目录
OVERWRITE_DRIVE_RESULT = True  # 若 Drive 聚合目录已存在, 是否覆盖


In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys


def run_command(command, cwd=None):
    print("$", " ".join(map(str, command)), flush=True)
    completed = subprocess.run(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError("命令失败: " + " ".join(map(str, command)))
    return completed



def parse_json_process_stdout(stdout_text):
    """解析仓库脚本输出的 JSON。

    脚本通常使用 indent=2 打印多行 JSON, 因此不能只读取最后一行。
    该函数从完整 stdout 中定位第一个 JSON 对象并解析, 便于 Colab cell 稳定接收结果。
    """
    resolved_text = str(stdout_text).strip()
    if not resolved_text:
        raise ValueError("脚本 stdout 为空, 无法解析 JSON 结果。")
    json_start = resolved_text.find("{")
    if json_start < 0:
        raise ValueError("脚本 stdout 中未找到 JSON 对象。")
    return json.loads(resolved_text[json_start:])

def split_paths(raw_paths):
    if isinstance(raw_paths, (list, tuple)):
        return [str(item).strip() for item in raw_paths if str(item).strip()]
    return [item.strip() for item in re.split(r"[;,]", str(raw_paths)) if item.strip()]


## 1. 挂载 Drive 并获取仓库

In [ ]:
try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
except Exception as exc:
    print(f"非 Colab 环境或 Drive 挂载失败: {exc}")

repo_path = Path(REPO_DIR)
if repo_path.exists():
    run_command(["git", "fetch", "--depth", "1", "origin", GIT_REF], cwd=repo_path)
    run_command(["git", "checkout", GIT_REF], cwd=repo_path)
    run_command(["git", "pull", "--ff-only", "origin", GIT_REF], cwd=repo_path)
else:
    run_command(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, REPO_DIR])
short_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True).strip()
print({"short_commit": short_commit})


## 2. 聚合 shard_runs

In [ ]:
resolved_shard_roots = split_paths(SHARD_ROOTS)
if not resolved_shard_roots:
    raise ValueError("SHARD_ROOTS 不能为空。请填入所有待聚合 shard 的目录或 zip 路径。")
family_id = f"real_video_vae_latent_probe_aggregated_{short_commit}"
local_family_root = Path(LOCAL_FAMILY_ROOT) / family_id
command = [
    sys.executable,
    "scripts/package_results/merge_real_video_vae_latent_shards.py",
    "--merged-run-root", MERGED_RUN_ROOT,
    "--family-root", str(local_family_root),
    "--protocol-config", PROTOCOL_CONFIG,
    "--mechanism-config", MECHANISM_CONFIG,
    "--runtime-profile", RUNTIME_PROFILE,
    "--short-commit", short_commit,
]
if EXCLUDE_LARGE_INTERMEDIATE_LATENTS:
    command.append("--exclude-large-intermediate-latents")
for shard_root in resolved_shard_roots:
    command.extend(["--shard-root", shard_root])
completed = run_command(command, cwd=REPO_DIR)
aggregation_summary = parse_json_process_stdout(completed.stdout)
print(aggregation_summary)


## 3. 落盘到 Drive shard_aggregated

In [ ]:
drive_family_root = Path(DRIVE_ROOT) / "TSTW" / "results" / "real_video_vae_latent_probe" / "shard_aggregated" / local_family_root.name
if drive_family_root.exists():
    if not OVERWRITE_DRIVE_RESULT:
        raise FileExistsError(drive_family_root)
    shutil.rmtree(drive_family_root)
drive_family_root.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(local_family_root, drive_family_root)
print({"drive_family_root": str(drive_family_root)})

# 使用阶段三的真实输入检查器验证聚合结果是否可被 baseline comparison 直接消费。
downstream_check_path = drive_family_root / "stage_two_package_for_baseline_check.json"
downstream_completed = run_command([
    sys.executable,
    "scripts/prepare_baselines/check_real_video_vae_package_for_baseline.py",
    "--package-root", str(drive_family_root),
    "--summary-path", str(downstream_check_path),
], cwd=REPO_DIR)
downstream_stage_two_check = parse_json_process_stdout(downstream_completed.stdout)
print({"downstream_stage_two_check": downstream_stage_two_check})


## 4. 最终完成性判断

In [ ]:
required_paths = {
    "family_summary": drive_family_root / "family_summary.json",
    "family_checks": drive_family_root / "family_checks.json",
    "family_manifest": drive_family_root / "family_manifest.json",
    "packages": drive_family_root / "packages",
}
completion_summary = {
    "result_flow": "shard_aggregate",
    "drive_family_root": str(drive_family_root),
    "required_paths": {name: path.exists() for name, path in required_paths.items()},
    "input_shard_roots": resolved_shard_roots,
    "ready_for_baseline_comparison_gate": bool(downstream_stage_two_check.get("package_ready_for_baseline_comparison")),
    "downstream_stage_two_check_path": str(downstream_check_path),
    "downstream_stage_two_check": downstream_stage_two_check,
}
completion_summary["status"] = all(completion_summary["required_paths"].values()) and completion_summary["ready_for_baseline_comparison_gate"]
print(json.dumps(completion_summary, ensure_ascii=False, indent=2))
if not completion_summary["status"]:
    raise RuntimeError(completion_summary)
